# Part e: Classification on MNIST


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

In [ ]:
import numpy as np
from src.data import load_mnist, one_hot, stratified_kfold_indices
from src.metrics import accuracy, confusion_matrix_multiclass, per_class_from_confusion, standard_error
from src.network import NeuralNetwork
from src.functions import softmax, ReLU, ReLU_der, identity_der, cross_entropy, softmax_cross_entropy_der
from src.optimizers import Adam
from src.plotting import heatmap

In [ ]:
x_train, y_train, x_test, y_test = load_mnist("../data/mnist.npz") # loading the mnist dataset, flattening the image pixel to shape (784,)
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape) # checking the shapes of training set, test set and true digits

rng = np.random.default_rng(2026) # setting a seed for reproducibility

# for every digit, find 1000 random positions of images that is this digit. Join the 1000 image positions for every digit in one long array
# This makes every digit equally represented
sub = np.concatenate([rng.choice(np.where(y_train == c)[0], 1000, replace=False) for c in range(10)])
X_sub, y_sub = x_train[sub], y_train[sub] # pick out these randomly chosen images by using their positions (sub)

# should give shape (10000, 784) since 1000 * 10 digit images and each flattened image has 784 pixels. 
# Bincount counts how many times each digit appear, should be 1000 of each digit:
print(X_sub.shape, np.bincount(y_sub))  # check shapes  

In [ ]:
from sklearn.metrics import accuracy_score

def cross_validate_by_hand(fit_predict, X, y, folds, score, scale=True):
    """One validation score and one training score per fold; the scaler is fitted on the training part of each fold only."""
    val_scores, train_scores = [], []
    for train_idx, val_idx in folds:
        X_tr, X_val = X[train_idx], X[val_idx]
        if scale:
            mean, std = X_tr.mean(axis=0), X_tr.std(axis=0)        # fitted on the training part ...
            X_tr, X_val = (X_tr - mean) / std, (X_val - mean) / std   # ... applied to both
        pred_val, pred_tr = fit_predict(X_tr, y[train_idx], X_val, X_tr)
        val_scores.append(score(y[val_idx], pred_val))
        train_scores.append(score(y[train_idx], pred_tr))
    return np.array(val_scores), np.array(train_scores)

In [ ]:
# Defining a function that trains a network and returns its predictions

def fit_predict_net(X_tr, y_tr, X_val, X_tr_again, gamma=1e-3, lmbda=0.0):
    '''
    Functions used:
    softmax turns the 10 outputs into probabilities that sum to 1.
    cross_entropy is the cost function for classification.
    softmax_cross_entropy_der is the derivative used in backpropagation for the last layer. 
    identity_der is a derivative that is always 1
    The Adam optimizer updates the weights during training.
    '''
    # Train our own NeuralNetwork (784 -> 100 -> 10, ReLU, softmax + cross entropy)
    net = NeuralNetwork(784, [100, 10], [ReLU, softmax], [ReLU_der, identity_der],
                        cross_entropy, softmax_cross_entropy_der, Adam(gamma=gamma),
                        regularization="L2", lmbda=lmbda)
    net.train(X_tr, one_hot(y_tr, 10), epochs=5, batch_size=100, rng=np.random.default_rng(2026))

    # Return the predicted digits (argmax of the softmax output)
    return np.argmax(net.predict(X_val), axis=1), np.argmax(net.predict(X_tr_again), axis=1)

In [ ]:
gammas, lambdas = [1e-4, 1e-3, 1e-2, 3e-2, 1e-1], [0.0, 1e-5, 1e-4, 1e-3]          # CHANGED: values for Adam on MNIST
folds = stratified_kfold_indices(y_sub, 5, np.random.default_rng(2026))  # CHANGED: stratified version, from the same file
table = np.zeros((len(gammas), len(lambdas)))
table_se = np.zeros_like(table)
for i, g in enumerate(gammas):
    for j, l in enumerate(lambdas):
        fp = lambda X_tr, y_tr, X_val, X_tr2, g=g, l=l: fit_predict_net(X_tr, y_tr, X_val, X_tr2, g, l)   # CHANGED: our fit_predict
        v, _ = cross_validate_by_hand(fp, X_sub, y_sub, folds, accuracy_score, scale=False)           # CHANGED: MNIST subset, accuracy, no scaling
        table[i, j], table_se[i, j] = v.mean(), v.std(ddof=1) / np.sqrt(len(v))
print("CV accuracy (rows gamma, columns lambda):\n", np.round(table, 4))
print("standard error:\n", np.round(table_se, 4))

# Visualize the results in a heatmap
heatmap(table, lambdas, gammas, r"$\lambda$", r"$\gamma$", label="CV accuracy", name="e_heatmap_l2")

In [ ]:
# choosing gamma and lambda: the best cell, and the one-standard-error rule (week 44 Tuesday, cv_grid)
shape = (len(gammas), len(lambdas))
i_best = np.unravel_index(np.argmax(table), shape)               # CHANGED: argmax (accuracy) instead of argmin (MSE)
within = table >= table[i_best] - table_se[i_best]               # CHANGED: >= best - s.e., instead of <= best + s.e.
# the one-standard-error rule: among the cells within one s.e., the largest penalty, then the smallest learning rate
cand = [(j, i) for i in range(shape[0]) for j in range(shape[1]) if within[i, j]]
j1, i1 = max(cand, key=lambda t: (lambdas[t[0]], -gammas[t[1]]))

print(f"best: gamma {gammas[i_best[0]]:g}, lambda {lambdas[i_best[1]]:g}: {table[i_best]:.4f} +/- {table_se[i_best]:.4f}")
print("within one standard error of the best:\n", within)
print(f"one-standard-error rule: gamma {gammas[i1]:g}, lambda {lambdas[j1]:g}: {table[i1, j1]:.4f}")

In [ ]:
# Now for logistic regression

def fit_predict_logreg(X_tr, y_tr, X_val, X_tr_again, gamma=1e-3, lmbda=0.0):
    # CHANGED from fit_predict_net: logistic regression, 784 -> 10 (no hidden layer), softmax + cross entropy
    net = NeuralNetwork(784, [10], [softmax], [identity_der],
                        cross_entropy, softmax_cross_entropy_der, Adam(gamma=gamma),
                        regularization="L2", lmbda=lmbda)
    net.train(X_tr, one_hot(y_tr, 10), epochs=5, batch_size=100, rng=np.random.default_rng(2026))
    return np.argmax(net.predict(X_val), axis=1), np.argmax(net.predict(X_tr_again), axis=1)

In [ ]:
# logistic regression baseline: the same cross-validation with fit_predict_logreg
#lambdas_lr = [0.0]                                                        # CHANGED: only gamma tuned (or use lambdas_lr = lambdas)
table_lr = np.zeros((len(gammas), len(lambdas)))
table_lr_se = np.zeros_like(table_lr)
for i, g in enumerate(gammas):
    for j, l in enumerate(lambdas):
        fp = lambda X_tr, y_tr, X_val, X_tr2, g=g, l=l: fit_predict_logreg(X_tr, y_tr, X_val, X_tr2, g, l)   # CHANGED: fit_predict_logreg
        v, _ = cross_validate_by_hand(fp, X_sub, y_sub, folds, accuracy_score, scale=False)
        table_lr[i, j], table_lr_se[i, j] = v.mean(), v.std(ddof=1) / np.sqrt(len(v))
print("logistic regression, CV accuracy (rows gamma, columns lambda):\n", np.round(table_lr, 4))
print("standard error:\n", np.round(table_lr_se, 4))

In [ ]:
# training on the whole dataset

# final models: trained once on all 60 000 training images, tested once on the 10 000 test images
pred_test_net, pred_train_net = fit_predict_net(x_train, y_train, x_test, x_train, gamma=0.01, lmbda=0.0)
pred_test_lr, pred_train_lr = fit_predict_logreg(x_train, y_train, x_test, x_train, gamma=0.01, lmbda=1e-4)

for name, pred in (("logistic regression", pred_test_lr), ("one hidden layer of 100", pred_test_net)):
    acc = accuracy_score(y_test, pred)
    print(f"{name}: test accuracy {acc:.4f} +/- {standard_error(acc, len(y_test)):.4f}")

In [ ]:

import matplotlib.pyplot as plt
from src.plotting import BLUE, RED, save_fig
from src.metrics import per_class_from_confusion      # CHANGED: import, since this is in plotting.py


def fig_confusion10(sm, name=None):                   # CHANGED: name instead of fname, as the other plotting functions
    """The 10 x 10 confusion matrix and the per-class precision and recall (week 43 Monday, fig_confusion10)."""
    C = sm["C"]
    prec, rec = per_class_from_confusion(C)
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.6), gridspec_kw={"width_ratios": [1.15, 1]})
    off = C.astype(float).copy()
    np.fill_diagonal(off, np.nan)
    im = ax[0].imshow(off, cmap="Reds", vmin=0, vmax=np.nanmax(off))
    for i in range(10):
        for j in range(10):
            if i == j:
                ax[0].text(j, i, f"{C[i, j]}", ha="center", va="center", fontsize=7, color=BLUE, fontweight="bold")
            elif C[i, j] > 0:
                ax[0].text(j, i, f"{C[i, j]}", ha="center", va="center", fontsize=7,
                           color="white" if C[i, j] > 0.6 * np.nanmax(off) else "black")
    ax[0].set_xticks(range(10))
    ax[0].set_yticks(range(10))
    ax[0].set_xlabel("predicted digit")
    ax[0].set_ylabel("true digit")
    ax[0].set_title(f"(a) 784-100-10 softmax network, test accuracy {sm['test_accuracy']:.4f}\n"
                    f"diagonal in blue; colour = off-diagonal count", fontsize=9)
    fig.colorbar(im, ax=ax[0], fraction=0.046)
    w = 0.38
    ax[1].bar(np.arange(10) - w / 2, prec, w, color=BLUE, label="precision (column)")
    ax[1].bar(np.arange(10) + w / 2, rec, w, color=RED, label="recall (row)")
    ax[1].set_ylim(0.9, 1.0)
    ax[1].set_xticks(range(10))
    ax[1].set_xlabel("digit")
    ax[1].legend(fontsize=8, loc="lower left")
    ax[1].set_title("(b) each class against the rest, from the same matrix", fontsize=9)
    fig.tight_layout()
    if name is not None:                              # CHANGED: the group's save_fig instead of fig.savefig + plt.close,
        save_fig(name)                                #          so the figure is also shown in the notebook
    return prec, rec

In [ ]:
#from src.plotting import fig_confusion10

C = confusion_matrix_multiclass(y_test, pred_test_net, 10)
sm = {"C": C, "test_accuracy": accuracy_score(y_test, pred_test_net)}
#prec, rec = fig_confusion10(sm, name="e_confusion_matrix")
prec, rec = fig_confusion10(sm)

# the largest confusions, from the same lecture cell
C_off = C.copy()
np.fill_diagonal(C_off, 0)
worst = np.dstack(np.unravel_index(np.argsort(-C_off.ravel())[:6], C_off.shape))[0]
print("largest confusions (true -> predicted: count): " + ", ".join(f"{i} -> {j}: {C_off[i, j]}" for i, j in worst))

In [ ]:
# Logistic regression vs scikit

from sklearn.linear_model import LogisticRegression
# Initialize the model (multinomial logistic regression with the saga solver)
model = LogisticRegression(solver='saga', max_iter=1000, random_state=42)
# Train the model
model.fit(x_train, y_train)
from sklearn.metrics import accuracy_score
# Make predictions on the test set
y_pred = model.predict(x_test)
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

Model Accuracy: 0.9257

In [ ]:
# comparing network with hidden layer with PyTorch

import time

def torch_models(kind="mlp"):
    """The 784-100-10 network of week 41 ('mlp') as nn.Sequential (week 44 Monday, torch_models)."""
    import torch.nn as nn
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 100), nn.ReLU(), nn.Linear(100, 10))   # CHANGED: only the 'mlp' model


def torch_mnist(kind="mlp", epochs=5, batch=64, lr=1e-3, seed=1, verbose=True):
    """Train one of torch_models on the 60 000 MNIST images with Adam; test accuracy after every epoch.

    The data are standardised with the MNIST mean and standard deviation
    (0.1307, 0.3081), as the chapter's transforms.Normalize does; the loss is
    nn.CrossEntropyLoss, which applies the softmax itself, so the network ends
    with logits.
    """
    import torch
    import torch.nn as nn
    torch.manual_seed(seed)
    Xtr, ytr, Xte, yte = x_train, y_train.astype(int), x_test, y_test.astype(int)   # CHANGED: our arrays instead of mnist_arrays()
    Xtr = torch.tensor(((Xtr - 0.1307) / 0.3081).astype("float32")); Ytr = torch.tensor(ytr)
    Xte = torch.tensor(((Xte - 0.1307) / 0.3081).astype("float32")); Yte = torch.tensor(yte)
    net = torch_models(kind)
    n_par = sum(q.numel() for q in net.parameters())
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    hist, t0 = [], time.time()
    for ep in range(1, epochs + 1):
        net.train()                                  # dropout on
        perm = torch.randperm(len(Xtr))
        run_loss, nb = 0.0, 0
        for i in range(0, len(Xtr) - batch + 1, batch):
            idx = perm[i:i + batch]
            opt.zero_grad(set_to_none=True)
            loss = lossf(net(Xtr[idx]), Ytr[idx])
            loss.backward()                          # Eqs. (10.29)-(10.33), by autograd
            opt.step()
            run_loss += loss.item(); nb += 1
        net.eval()                                   # dropout off
        with torch.no_grad():
            pred = torch.cat([net(Xte[i:i + 1000]).argmax(1) for i in range(0, len(Xte), 1000)])
            acc = (pred == Yte).double().mean().item()
        hist.append((ep, run_loss / nb, acc, time.time() - t0))
        if verbose:
            print(f"  torch {kind:3s} epoch {ep}: loss {run_loss / nb:.4f}  test accuracy {acc:.4f}  ({time.time() - t0:.0f} s)")
    return {"history": np.array(hist), "n_params": n_par, "test_accuracy": hist[-1][2], "pred": pred.numpy()}   # CHANGED: removed the CNN-only outputs

In [ ]:
r = torch_mnist("mlp")
print(f"  -> torch mlp: {r['n_params']} parameters, final test accuracy {r['test_accuracy']:.4f}")

# Logistic regression network, 784 -> 10 (no hidden layers), softmax output + cross entropy
'''
Functions used:

softmax turns the 10 outputs into probabilities that sum to 1.
cross_entropy is the cost function for classification.
softmax_cross_entropy_der is the derivative used in backpropagation for the last layer. 
identity_der is a derivative that is always 1
The Adam optimizer updates the weights during training.
'''
# Creating the network
logreg = NeuralNetwork(784, [10], [softmax], [identity_der],      # identity_der: the softmax derivative is already in softmax_cross_entropy_der
                       cross_entropy, softmax_cross_entropy_der, Adam(gamma=0.001))

# Training the network with stochastic gradient descent, gives the training cost after each epoch
history = logreg.train(x_tr, Y_tr, epochs=5, batch_size=100, rng=np.random.default_rng(2026))


# Finding the predicted numbers for each image in the validation dataset
# logreg.predict(x_val) gives 10 probabilities for each of the 10 000 validation images, with shape (10000, 10).
# np.argmax(..., axis=1) picks, for each image, the position of the largest probability, i.e. the predicted digit. The result has shape (10000,).
pred_val = np.argmax(logreg.predict(x_val), axis=1)    # predicted digit = the output with the highest probability

# compares the predicted digits with the true digits y_val, and prints the fraction that are correct
print("validation accuracy for logistic regression:", accuracy(y_val, pred_val))

# Network with one hidden layer of 100 ReLU nodes: 784 -> 100 -> 10
# same principle as for logistic regression, but adding activation function ReLU for the hidden layer and outputs 100 and 10

net_layer = NeuralNetwork(784, [100, 10], [ReLU, softmax], [ReLU_der, identity_der],
                    cross_entropy, softmax_cross_entropy_der, Adam(gamma=0.001))   # a new Adam for the new network

history_net_layer = net_layer.train(x_tr, Y_tr, epochs=5, batch_size=100, rng=np.random.default_rng(2026))

pred_val_net_layer = np.argmax(net_layer.predict(x_val), axis=1)
print("validation accuracy:", accuracy(y_val, pred_val_net_layer))